<a href="https://colab.research.google.com/github/subaina705/mentah-health-support-chatbot/blob/main/Mental_Health_Support_Chatbot_(Fine_Tuned).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mental Health Support Chatbot

# Dataset Note
The original empathetic_dialogues dataset script is no longer supported by newer versions of the HuggingFace datasets library. An equivalent mirror dataset (bdotloh/empathetic-dialogues-contexts) was used, which contains the same emotional situations and context data in a compatible Parquet format.

In [4]:
!pip install transformers datasets accelerate torch -q

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset first
dataset = load_dataset("bdotloh/empathetic-dialogues-contexts")

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def format_conversation(example):
    prompt = (
        f"Emotion: {example['emotion']}\n"
        f"Situation: {example['situation']}\n"
        f"Bot: I hear you. That sounds really {example['emotion']}. "
        f"It's completely okay to feel this way."
        f"{tokenizer.eos_token}"
    )
    return {"text": prompt}

formatted = dataset["train"].map(format_conversation)

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = formatted.map(tokenize, batched=True, remove_columns=formatted.column_names)

print("✅ Dataset ready! Sample count:", len(tokenized))
print("\nSample formatted text:")
print(formatted[0]["text"])

README.md:   0%|          | 0.00/540 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19209 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2756 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2542 [00:00<?, ? examples/s]

Map:   0%|          | 0/19209 [00:00<?, ? examples/s]

Map:   0%|          | 0/19209 [00:00<?, ? examples/s]

✅ Dataset ready! Sample count: 19209

Sample formatted text:
Emotion: sentimental
Situation: I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.
Bot: I hear you. That sounds really sentimental. It's completely okay to feel this way.<|endoftext|>


In [7]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

print(" Model loaded! Parameters:", model.num_parameters())

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

 Model loaded! Parameters: 81912576


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mental-health-bot",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    fp16=True,
    warmup_steps=100,
    prediction_loss_only=True,
)

print("Training config ready!")

Training config ready!


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

print("Starting fine-tuning")
trainer.train()
print(" Training complete!")

Starting fine-tuning


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.576232
200,1.539910
300,1.520972
400,1.545244
500,1.476041
600,1.471467
700,1.483085
800,1.454534
900,1.467669
1000,1.423833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
100,2.576232
200,1.539910
300,1.520972
400,1.545244
500,1.476041
600,1.471467
700,1.483085
800,1.454534
900,1.467669
1000,1.423833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
model.save_pretrained("./mental-health-bot-final")
tokenizer.save_pretrained("./mental-health-bot-final")
print(" Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Model saved!


In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def chat(user_input):
    prompt = f"Emotion: neutral\nSituation: {user_input}\nBot:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)  # <-- this line is the fix

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    bot_reply = response.split("Bot:")[-1].strip()
    return bot_reply

print("Chat function ready! Running on:", device)

Chat function ready! Running on: cuda


In [12]:
print("=" * 50)
print("   💚 Mental Health Support Chatbot")
print("   Type 'quit' to exit")
print("=" * 50)

while True:
    user_input = input("\n👤 You: ").strip()

    if user_input.lower() in ["quit", "exit", "bye"]:
        print("🤖 Bot: Take care of yourself. You're not alone. 💚")
        break

    if not user_input:
        continue

    print("🤖 Bot:", chat(user_input))

   💚 Mental Health Support Chatbot
   Type 'quit' to exit

👤 You: i am feeling lonely
🤖 Bot: I hear you. That sounds reallyutral... It's completely okay to feel this way, but it's completely okay to express yourself as if you're not alone in your life anymore? Or is that just a feeling of rejection and shame?
Bot at 2:00pm on my next date!

👤 You: quit
🤖 Bot: Take care of yourself. You're not alone. 💚
